## Étude du perceptron binaire

In [1]:
import numpy as np
from itertools import product
rng = np.random.default_rng(0)
import matplotlib.pyplot as plt

On définit ici l'énergie, qui permet d'évaluer la réussite d'un vecteur de poids proposée par un perceptron. Plus l'énergie est haute, plus le nombre de contraintes violées est haut aussi. A l'inverse, une énergie égale à zéro suggère que le vecteur de poids proposé est une solution du problème. 

In [2]:
def energy (w, x, sigma):
    """
    w : vecteur des poids
    x : matrices des motifs
    sigma : vecteur de contrainte
    """
    x = np.array(x)
    w = np.array(w)
    sigma = np.array(sigma)

    ## On fait tous les tests en faisant le produit matriciel :
    tab = np.dot(x, w)
    
    energy = 0
    for i in range(len(tab)):
        ## Chaque fois qu'un test est différent de son étiquette, la contrainte n'est pas vérifiée :
        if sigma[i]*tab[i] <= 0:
            energy+=1
    return energy

Le premier algorithme testé est un algorithme glouton. Le concept est simple. Il change les poids un par un et accepte les modifications seulement si celles-ci engendre une baisse de l'énergie. Toutefois, comme expliqué dans le PDF, les résultats ne sont pas satisfaisants. Ainsi, on adoucit la règle dans le deuxième algorithme en tolérant des échanges de spins qui garde la même énergie. Pour ne pas avoir une boucle infinie, nous imposons toutefois un nombre maximal de changement égal à 1000.

In [3]:
def glouton(w, x, sigma, max_passage = 1000):
    """
    """
    w = np.array(w).copy()
    x = np.array(x)
    sigma = np.array(sigma)
    E = energy(w, x, sigma)
    print("Départ : E = ", E)
    for passage in range(max_passage):
        spin_acceptes = 0
        for i in range (len(w)):
            w[i] = -w[i]
            E_essai = energy(w, x, sigma)
            if E_essai < E:
                spin_acceptes+=1
                E = E_essai
                if E == 0:
                    return E, w
            else:
                w[i] = -w[i]
        print(spin_acceptes, E)

        if spin_acceptes == 0:
            return E, w
    return E, w

def glouton_hist(w, x, sigma, max_passage=1000):
    w = np.array(w).copy()
    E = energy(w, x, sigma)
    hist = [E]
    for passage in range(max_passage):
        acceptes = 0
        for i in range(len(w)):
            w[i] = -w[i]
            E_essai = energy(w, x, sigma)
            if E_essai < E:
                acceptes += 1
                E = E_essai
                hist.append(E)
                if E == 0:
                    return E, w, hist
            else:
                w[i] = -w[i]
        if acceptes == 0:
            return E, w, hist
    return E, w, hist
    

Lors des tests, nous voulions mesurer l'énergie minimale qu'est capable de trouver les algorithmes précédemment définis en fonciton du nombre de contrainte. Cela permet de visualiser l'écart par rapport à la valeur limite théorique. Pour cela, pour chaque instances tirées, nous avons relancé l'algorithme plusieurs fois à partir de différents vecteurs de poids tirés aléatoirement, afin de maximiser la chance de trouver une énergie basse voire nulle. 

In [4]:
def glouton_restart(w, x, sigma, n_restart = 10):
    current_energy, current_w = glouton(w, x, sigma)
    for i in range(n_restart):
        w_new = rng.choice([-1, 1], size=N)
        new_energy, w_opt = glouton(w_new, x, sigma)
        if new_energy < current_energy :
            current_energy = new_energy
            current_w = w_opt
    return current_energy, current_w

Nous avons ensuite appliqué cette algorithme à différentes valeurs de alpha afin de voir l'influence de ce dernier sur l'énergie et de pouvoir comparer avec la valeur théorique limite donnée par le calcul de Gardner. 

In [1]:
## On va faire un balayage en alpha = P/n:
def balayage(P_start,P_end, gap, N):
    compt = []
    energ = []
    P = np.array([i for i in range(P_start,P_end, gap)])
    for i in range(P_start, P_end, gap):
        compteur = 0
        energie = 0
        for k in range(50):
            w0 = rng.choice([-1, 1], size=N)
            x = []
            for p in range(i):
                x.append(rng.choice([-1, 1], size=N))
            sigma = rng.choice([-1, 1], size=i)
            E = glouton_restart_patient(w0, x, sigma)[0]
            if E == 0:
                compteur += 1
            energie += E
        compt.append(compteur)
        energ.append(energie/50)
    return np.array(compt), P, np.array(energ)

Enfin, pour les mêmes raisons que précedemment, nous avons opté pour un algorithme qui tolérait les échanges de spins ne faisant pas varier l'énergie. 

In [1]:
def glouton_patient(w, x, sigma, max_passage=1000, patience=30):
    w = np.array(w).copy()
    x = np.asarray(x)
    sigma = np.asarray(sigma)

    h = x @ w                              # marges non signées, taille P
    E = int(np.sum(sigma * h <= 0))
    sans_gain = 0

    for passage in range(max_passage):
        E_debut = E
        for i in range(len(w)):
            h_essai = h - 2 * w[i] * x[:, i]       # mise à jour incrémentale
            E_essai = int(np.sum(sigma * h_essai <= 0))
            if E_essai <= E:
                w[i] = -w[i]
                h = h_essai
                E = E_essai
                if E == 0:
                    return E, w
        if E == E_debut:
            sans_gain += 1
            if sans_gain >= patience:
                return E, w
        else:
            sans_gain = 0

    return E, w

In [7]:
def glouton_restart_patient(w, x, sigma, n_restart = 10):
    current_energy, current_w = glouton_patient(w, x, sigma)
    for i in range(n_restart):
        w_new = rng.choice([-1, 1], size=N)
        new_energy, w_opt = glouton_patient(w_new, x, sigma)
        if new_energy < current_energy :
            current_energy = new_energy
            current_w = w_opt
    return current_energy, current_w

La partie suivante portait sur l'étude du voisinage. Pour cela, nous listions toutes les solutions et tirions une solution au hasard. Nous comptions ensuite le nombre de solutions au voisinage de cette solution et comparions avec une solution donnée par notre algorithme. 

In [8]:
## Pour calculer théoriquement le voisinage d'une solution, il faut le comparer avec l'algorithme :
rng = np.random.default_rng(0)

N, P = 20, 12
x = rng.choice([-1, 1], size=(P, N))
sigma = rng.choice([-1, 1], size=P)

def toutes_solutions(N, sigma, x):
    
    W = np.array(list(product([-1, 1], repeat=N)))
    
    marges = (W @ x.T) * sigma

    """
    On peut remplacer cette boucle par axis = 1 --> plus rapide.
    E = []
    for k in range(len(marges)):
        E += [np.sum(marges[k] <= 0)]
    """
    E = np.sum(marges <= 0, axis = 1)
    
    return W[E == 0]

sols = toutes_solutions(N, sigma, x)

print("Partie théorique :")
print(f"N = {N}, P = {P}, alpha = {P/N}")
print(f"nombre de solutions : {len(sols)} sur {2**N} configurations")


def voisin_sols(sol, solutions, distance):
    nb_voisins = 0
    for element in solutions:
        tab = (sol-element)**2
        somme = np.sum(tab)
        if somme <= 4*distance : 
            nb_voisins += 1
    return nb_voisins - 1 


print("Partie expérimentale :")
## Comparaison avec l'algorithme : 
N, P = 20, 8
w = rng.choice([-1, 1], size=N)
E, sol_exp = glouton_patient(w, x, sigma, max_passage=1000, patience=30)
print(E, sol_exp)


print("Comparaison :")
distance = 2

k = rng.integers(0, len(sols))


sol_th = sols[k]
nb_th = voisin_sols(sol_th, sols, distance)

nb_exp = voisin_sols(sol_exp, sols, distance)

print(f"Le nombre de solutions à une distance {distance} de la solution aléatoire est : {nb_th}")
print(f"Le nombre de solutions à une distance {distance} de la solution gloutonne est : {nb_exp}")

Partie théorique :
N = 20, P = 12, alpha = 0.6
nombre de solutions : 16 sur 1048576 configurations
Partie expérimentale :
5 [-1  1 -1 -1 -1 -1  1 -1  1  1  1  1  1  1 -1  1 -1 -1 -1 -1]
Comparaison :
Le nombre de solutions à une distance 2 de la solution aléatoire est : 8
Le nombre de solutions à une distance 2 de la solution gloutonne est : -1


In [9]:
## On fait des moyennes pour comparer :
distance = 2
n_ech = 50

# population algorithmique
nb_exp = []
while len(nb_exp) < n_ech:
    w0 = rng.choice([-1, 1], size=N)
    E, s = glouton_patient(w0, x, sigma, max_passage=1000, patience=30)
    if E == 0:
        nb_exp.append(voisin_sols(s, sols, distance))

# population uniforme
idx = rng.integers(0, len(sols), size=n_ech)
nb_th = [voisin_sols(sols[k], sols, distance) for k in idx]

print(f"solutions aléatoires : {np.mean(nb_th):.1f}")
print(f"solutions gloutonnes : {np.mean(nb_exp):.1f}")

solutions aléatoires : 3.8
solutions gloutonnes : 4.7
